# Explore the zoo results
Reads what `mmd analyse` and `mmd zoo` wrote to the store (`results/<cohort_version>/<model>/<pair>/`, figures under
`figures/` with the same layout). This notebook has no GPU or model code. Run it from the repo with the core
environment (`uv sync`, then pick the `.venv` kernel), or in Colab after `pip install -e .`.

Settings come from `mmd.toml` / `MMD_*`, the same as the CLI.

In [ ]:
from google.auth import default  # noqa: F401  (gcsfs uses your gcloud / Colab credentials)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mmd import config, fsio

cfg = config.load()
PAIR = 'cxr_notes'                       # or ecg_notes
models = sorted(p.rstrip('/').split('/')[-2] for p in fsio.glob(cfg._p('results', cfg.cohort_version, '*', PAIR))
                if '/zoo/' not in p)
print(cfg.root, cfg.cohort_version, PAIR, models)

## 1. The zoo table
One row per model × tier × readout × classifier:
- `info_share_note`: the note's share of the outcome information, from the two single-modality arms;
- `reliance_share_note`: the note's share of the counterfactual |Δ risk|;
- `bias` = reliance − information. Above 0 means the model leans on the note more than the note's information justifies.

`subset = official split` repeats every model on stays whose baseline film is outside the MIMIC-CXR train split,
which matters for MIMIC-exposed models.

In [ ]:
zoo = fsio.read_parquet(cfg._p('results', cfg.cohort_version, 'zoo', f'zoo_{PAIR}.parquet'))
cols = ['model', 'fusion', 'domain', 'mimic_exposure', 'subset', 'level_id', 'readout', 'clf', 'auroc_signal_only',
        'auroc_note_only', 'auroc_both', 'info_share_note', 'reliance_share_note', 'reliance_lo', 'reliance_hi', 'bias']
zoo[[c for c in cols if c in zoo]].round(3)

## 2. Reliance against information
Each point is one model, using the raw P(Yes) readout (behaviour) on the chosen tier. Points above the diagonal lean on
the note more than its information share justifies. Filled markers have no known MIMIC exposure; hollow markers were
trained on MIMIC data.

In [ ]:
LEVEL, CLF, SUBSET = 'L1', 'no probe (raw P(Yes))', 'all'
t = zoo[(zoo.level_id == LEVEL) & (zoo.clf == CLF) & (zoo.subset == SUBSET)].dropna(subset=['reliance_share_note'])
fig, ax = plt.subplots(figsize=(5.5, 5.2))
ax.plot([0, 1], [0, 1], color='#bdbdbd', ls='--', lw=1)
for r in t.itertuples():
    exposed = r.mimic_exposure not in (None, 'none')
    ax.errorbar(r.info_share_note, r.reliance_share_note,
                yerr=[[r.reliance_share_note - r.reliance_lo], [r.reliance_hi - r.reliance_share_note]],
                fmt='o', ms=7, capsize=3, mfc='white' if exposed else '#2a78d6', color='#2a78d6')
    ax.annotate(r.model, (r.info_share_note, r.reliance_share_note), textcoords='offset points', xytext=(6, 4),
                fontsize=8)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('note share of the outcome information (single-modality arms)')
ax.set_ylabel('note share of the counterfactual |Δ risk|')
ax.set_title(f'{LEVEL} · {CLF} · {SUBSET}', loc='left', fontsize=10)
plt.show()

## 3. Per-model tables
The same tables the notebooks write (`swap4_*`), for any model in the store.

In [ ]:
MODEL = models[0]
d = cfg.results_dir(MODEL, PAIR)
summary = fsio.read_parquet(f'{d}/swap4_summary_{PAIR}_clean.parquet')
auroc = fsio.read_parquet(f'{d}/swap4_auroc_{PAIR}.parquet')
paired = fsio.read_parquet(f'{d}/swap4_paired_dominance_{PAIR}.parquet')
by_draw = fsio.read_parquet(f'{d}/swap4_auroc_by_draw_{PAIR}.parquet')
summary[['level_id', 'readout', 'clf', 'family', 'slot', 'n', 'delta_died', 'delta_survived', 'gap', 'gap_lo',
         'gap_hi']].round(4)

In [ ]:
# single donor vs mean of donors (the matched-swap AUROC artefact): report the single-donor numbers
kind = np.where(by_draw.scored == 'mean of draws', 'mean of draws', 'one donor (avg over draws)')
by_draw.assign(kind=kind).pivot_table(index=['readout', 'level_id', 'family', 'slot'], columns='kind',
                                      values='d_auroc').round(3)

## 4. Figures
The charts `mmd analyse` saved for a model.

In [ ]:
from IPython.display import Image, display
figs = fsio.glob(f'{cfg.fig_dir(MODEL, PAIR)}/*.png')
print(len(figs), 'figures')
for p in figs[:4]:
    with fsio.open_(p) as f:
        display(Image(f.read()))

## Results after the modality-set refactor
`mmd analyse` now writes plain table names into `results/<cohort_version>/<model>/<modset>/` (`summary_clean`, `auroc`, `reliance`, `pairwise_dominance`, `informativeness`, ...) and the zoo table to `results/<cohort_version>/zoo/<modset>`. The cells above read the older `swap4_*_<pair>` files; this cell reads the new ones. `MODSET` can be any set from `mmd modsets` (`cxr_notes`, `ecg_notes`, or one you defined).

In [ ]:
from mmd.zoo import Zoo

MODSET = 'cxr_notes'
zoo = Zoo(cfg)
zoo_tab = zoo.table(modset=MODSET, save=False)          # info / reliance / bias per model and modality
display(zoo_tab[['model', 'level_id', 'readout', 'clf', 'mod', 'auroc_only', 'info_share', 'reliance_share', 'bias']].round(3)
        if len(zoo_tab) else 'no results for this set yet')

res = {}
for m in [m for m, A in zoo.models().items() if zoo.sets[MODSET].fits(A.modalities)]:
    d = cfg.results_dir(m, MODSET)
    if fsio.exists(f'{d}/model.json'):
        res[m] = {t: fsio.read_parquet(f'{d}/{t}.parquet') for t in ('summary_clean', 'auroc', 'reliance', 'informativeness')}
print({m: {t: len(v) for t, v in r.items()} for m, r in res.items()})

## Discordance: how much each modality can mislead a model, against a late-fusion benchmark

Written by `mmd discordance` (`mmd/analysis/discordance.py`). Every model of a modality set is analysed on the SAME
patients (`results/zoo/discordance_cohort_<set>`), every condition on the same patients, and a donor condition's AUROC
is the mean over its 5 donor draws of one AUROC per draw. One row per readout x classifier: the model's own answer
(`answer`), and each hidden-state readout probed with logistic regression (and XGBoost where installed).

- **capture** = AUROC(all) − 0.5; `low_capture` below 0.05 (every ratio divides by it)
- **necessity** = (AUROC(all) − AUROC(m removed)) ÷ capture
- **discordance sensitivity** = (AUROC(all) − AUROC(m from an opposite-outcome donor)) ÷ capture
- **substitution effect** = AUROC(all) − AUROC(m from a same-outcome donor): negative control, ≈ 0
- **late-fusion benchmark**: the average log-odds of calibrated single-modality probes, put through the same
  perturbations; `expected_*` are its values, `benchmark_valid` = it reaches the best single modality's AUROC
- **excess discordance sensitivity** = discordance sensitivity − expected: > 0, m misleads the model more than the
  model's own evidence from m justifies (the headline)
- **modality bias score** = conflict share − expected conflict share; **total modality bias** = ½ Σ |score|
- **demographic AUROC**: age + sex alone on the same patients, what any model should beat

In [ ]:
from mmd.zoo import Zoo

zoo = Zoo(cfg)
disc = zoo.discordance_table(compute=False, save=False)   # compute=True recomputes from the stored records (CPU)
disc_mod, disc_model = disc.per_mod, disc.per_model
display(disc_model[['model', 'modset', 'level_id', 'readout', 'clf', 'n', 'died', 'auroc_reference',
                    'demographic_auroc', 'capture', 'low_capture', 'benchmark_auroc_reference', 'benchmark_valid',
                    'total_modality_bias', 'dominant_modality', 'max_substitution_effect']].round(3))
display(disc_mod[['model', 'level_id', 'readout', 'clf', 'mod', 'auroc_reference', 'auroc_without',
                  'auroc_counterfactual', 'necessity', 'discordance_sensitivity', 'expected_discordance_sensitivity',
                  'excess_discordance_sensitivity', 'substitution_effect', 'modality_bias_score']].round(3))

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

LEVEL = 'L3'                      # opentslm-sp has L1 only so far; at L1 every model has low capture
READOUT, CLF = 'answer', 'no probe (answer log-odds)'   # or e.g. 'answer mean state', 'Logistic regression'
INK, INK2, MUTED, GRID, SURFACE = '#0b0b0b', '#52514e', '#8a8984', '#e4e3df', '#fcfcfb'
MOD_ORDER = ['e', 'x', 'n']
MOD_NAME = {'e': 'ECG', 'x': 'Chest X-ray', 'n': 'Note'}
MOD_COLOUR = {'x': '#2a78d6', 'e': '#eb6834', 'n': '#1baf7a'}          # categorical slots 1-3 (validated all-pairs)
MODEL_NAME = {'gemma3-4b': 'Gemma 3 4B', 'llava-med': 'LLaVA-Med', 'qwen2.5-vl-7b': 'Qwen2.5-VL 7B',
              'opentslm-sp': 'OpenTSLM-SP', 'opentslm-sp-gemma': 'OpenTSLM-SP (Gemma)'}
DIVERGING = LinearSegmentedColormap.from_list('gap', ['#2a78d6', '#f0efec', '#e34948'])
rc = {'font.size': 10, 'axes.edgecolor': GRID, 'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2,
      'axes.spines.top': False, 'axes.spines.right': False, 'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE}

sel = lambda d: d[(d.level_id == LEVEL) & (d.dose == 100) & (d.clf == CLF)
                  & ((d.readout == READOUT) | d.readout.str.startswith(READOUT + ' ('))]   # OpenTSLM: '... (SP)'
pm, mm = sel(disc_mod).copy(), sel(disc_model).copy()
gap = pm.groupby(['model', 'modset']).excess_discordance_sensitivity.max().rename('max_gap').reset_index()
mm = mm.merge(gap, on=['model', 'modset'], how='left').sort_values('max_gap', ascending=False)
rows = [(r.model, r.modset) for r in mm.itertuples()]
low = {(r.model, r.modset) for r in mm.itertuples() if r.low_capture or not r.interpretable or not r.benchmark_valid}
row_label = {(r.model, r.modset): f"{MODEL_NAME.get(r.model, r.model)}\n{' + '.join(MOD_NAME[m] for m in r.mods.split('+'))}"
             for r in mm.itertuples()}
mods = [m for m in MOD_ORDER if m in set(pm['mod'])]

In [ ]:
# necessity (hollow: modality removed) vs discordance sensitivity (filled: modality conflicts), per model x modality
pairs = [(r, m) for r in rows for m in mods if ((pm.model == r[0]) & (pm.modset == r[1]) & (pm['mod'] == m)).any()]
with plt.rc_context(rc):
    fig, ax = plt.subplots(figsize=(8.2, 0.42 * len(pairs) + 1.6))
    ylab = []
    for i, ((model, ms), m) in enumerate(pairs):
        r = pm[(pm.model == model) & (pm.modset == ms) & (pm['mod'] == m)].iloc[0]
        c = MOD_COLOUR[m]
        ax.plot([r.necessity, r.discordance_sensitivity], [i, i], color=c, linewidth=2, solid_capstyle='round', zorder=1)
        ax.scatter([r.necessity], [i], s=64, facecolor=SURFACE, edgecolor=c, linewidth=2, zorder=2)
        ax.scatter([r.discordance_sensitivity], [i], s=64, color=c, edgecolor=SURFACE, linewidth=2, zorder=3)
        ax.scatter([r.expected_discordance_sensitivity], [i], s=140, marker='|', color=INK, linewidth=2, zorder=4)
        ylab.append(f"{MODEL_NAME.get(model, model)} · {MOD_NAME[m]}" + ('  (flagged)' if (model, ms) in low else ''))
    ax.axvline(0, color=GRID, linewidth=1, zorder=0)
    ax.axvline(1, color=MUTED, linewidth=1, linestyle=(0, (3, 3)), zorder=0)
    ax.set_yticks(range(len(pairs)), ylab)
    ax.invert_yaxis()
    ax.set_xlabel('share of above-chance performance lost  (dashed line at 1 = down to chance)\n'
                  '(AUROC drop ÷ (reference AUROC − 0.5))')
    ax.grid(axis='x', color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', length=0)
    ax.scatter([], [], s=64, facecolor=SURFACE, edgecolor=INK2, linewidth=2, label='necessity: modality removed')
    ax.scatter([], [], s=64, color=INK2, label='discordance sensitivity: modality conflicts')
    ax.scatter([], [], s=140, marker='|', color=INK, linewidth=2, label='expected (late-fusion benchmark)')
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=3, frameon=False, fontsize=9)
    fig.suptitle(f'Needed vs misleading, against the late-fusion benchmark ({LEVEL}, {READOUT})', x=0.02, ha='left',
                 color=INK, fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    plt.show()

In [ ]:
# excess discordance sensitivity (model − late-fusion benchmark) per model x modality; capture on the right
M = np.full((len(rows), len(mods)), np.nan)
for r in pm.itertuples():
    if (r.model, r.modset) in rows:
        M[rows.index((r.model, r.modset)), mods.index(r.mod)] = r.excess_discordance_sensitivity
VMAX = 2.0
with plt.rc_context(rc):
    fig, ax = plt.subplots(figsize=(2.2 + 1.5 * len(mods) + 2.6, 0.8 * len(rows) + 1.9))
    ax.imshow(np.ma.masked_invalid(np.clip(M, -VMAX, VMAX)), cmap=DIVERGING, vmin=-VMAX, vmax=VMAX, aspect='auto')
    for i in range(len(rows)):
        for j in range(len(mods)):
            v = M[i, j]
            ax.text(j, i, '–' if np.isnan(v) else f'{v:+.2f}', ha='center', va='center',
                    color=MUTED if np.isnan(v) else (SURFACE if abs(v) > 1.2 else INK), fontsize=11)
        r = mm.iloc[i]
        txt = f"{r.capture:.3f}" + ('  low' if r.low_capture else '') + ('' if r.benchmark_valid else '  benchmark n/a')
        ax.text(len(mods) - 0.4, i, txt, ha='left', va='center', color=INK2 if r.low_capture else INK, fontsize=10)
    ax.text(len(mods) - 0.4, -0.75, 'capture', ha='left', va='center', color=INK2, fontsize=9)
    ax.set_xticks(range(len(mods)), [MOD_NAME[m] for m in mods])
    ax.set_yticks(range(len(rows)), [row_label[r] for r in rows])
    ax.xaxis.tick_top()
    ax.tick_params(which='both', length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_xticks(np.arange(-.5, len(mods)), minor=True)
    ax.set_yticks(np.arange(-.5, len(rows)), minor=True)
    ax.grid(which='minor', color=SURFACE, linewidth=2)
    ax.set_xlim(-.5, len(mods) + 1.2)
    fig.suptitle(f'Excess discordance sensitivity by model and modality ({LEVEL}, {READOUT})', x=0.02, ha='left',
                 color=INK, fontsize=12)
    fig.text(0.02, 0.01, 'discordance sensitivity − the late-fusion benchmark\'s.  red: m misleads the model more than '
             'its own evidence from m justifies; blue: the model discounts m.\n– : modality not taken.  capture = '
             'reference AUROC − 0.5 ("low" below 0.05: ratios are noisy).', color=INK2, fontsize=8, ha='left')
    fig.tight_layout(rect=(0, 0.07, 1, 0.95))
    plt.show()